# Imports necessários

In [1]:
import pandas as pd 
import numpy as np
import json


# Obtenção dos dados

In [2]:
with open("../dados/dados_nivel_1.json", "r", encoding="utf-8") as f:
    dados = json.load(f)

df = pd.DataFrame(dados)

In [3]:
df.head()

,taxa_cambio_usd_brl,operacoes
0,5.4,"{'id': 'OP-0001', 'cliente_id': 'CLI-A-1', 'da..."
1,5.4,"{'id': 'OP-0002', 'cliente_id': 'CLI-A-1', 'da..."
2,5.4,"{'id': 'OP-0003', 'cliente_id': 'CLI-A-1', 'da..."
3,5.4,"{'id': 'OP-0004', 'cliente_id': 'CLI-A-1', 'da..."
4,5.4,"{'id': 'OP-0005', 'cliente_id': 'CLI-A-2', 'da..."


In [4]:
df['taxa_cambio_usd_brl'].nunique()

1

#### Observações: 
* coluna 'taxa_cambio_usd_brl' possui um único valor, enquanto 'operacoes' carrega toda a estrutura dos dados
* vamos criar uma variável para guardar o valor da taxa de cambio

In [5]:
TAXA_CAMBIO = float(df["taxa_cambio_usd_brl"].iloc[0])

* Na coluna 'operacoes', todos os registros possuem a mesma estrutura? Vamos testar com a primeira linha

In [6]:
# compara as chaves da primeira linha com as demais, para verificar se todas possuem a mesma estrutura
chaves_esperadas = set(dados["operacoes"][0].keys())

for i, operacao in enumerate(dados["operacoes"]):
    chaves = set(operacao.keys())
    
    if chaves != chaves_esperadas:
        print(f"Operação {i} possui estrutura diferente:")
        print("Faltando:", chaves_esperadas - chaves)
        print("A mais:", chaves - chaves_esperadas)

#### Todos os registros possuem as mesmas chaves, portanto, podemos criar o DataFrame somente com a coluna 'operacoes'.

In [7]:
with open("../dados/dados_nivel_1.json", "r", encoding="utf-8") as f:
    dados = json.load(f)


df = pd.DataFrame(dados["operacoes"])

In [8]:
# como é um dataframe pequeno, prefiro visualizá-lo todo.
df

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


#### Pontos de qualidade dos dados:
* dados nos formatos adequados? existem Nans? existem duplicatas?
* ID's são únicos?
* datas estão todas no mesmo formato? são todas datas válidas?
* existem dados diferentes que representam a mesma coisa? por exemplo: pix e PIX, deposito e depósito...


# Entendimento da Base

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           20 non-null     object
 1   cliente_id   20 non-null     object
 2   data         19 non-null     object
 3   valor        20 non-null     int64 
 4   moeda        20 non-null     object
 5   canal        20 non-null     object
 6   tipo         20 non-null     object
 7   contraparte  20 non-null     object
 8   observacao   20 non-null     object
dtypes: int64(1), object(8)
memory usage: 1.5+ KB


In [10]:
df[df.duplicated(keep=False)]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


In [11]:
df[df['data'].isna()]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
17,OP-0017,CLI-A-5,None,4300,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema


* há um registro duplicado de id = OP-0007 --> como todos os dados são os mesmos, removeremos a duplicata;
* existe uma data Nan, com observação de 'data nao capturada pelo sistema'. o que fazer com esse registro? mantemos, mas ficará sob nosso radar;
* 'valor' está como inteiro, mas podemos transforma-lo para float;
* a seguir vamos fazer as demais verificações/limpezas na base.

In [12]:
df = df.drop_duplicates(keep="first")

In [13]:
df['valor'] = df['valor'].astype(float)

C:\Users\yghor\AppData\Local\Temp\ipykernel_6624\1997309100.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['valor'] = df['valor'].astype(float)


In [14]:
df.shape

(19, 9)

## id

In [15]:
# 19 id's únicos, assim como 19 registros na base.
df['id'].nunique()

19

## cliente_id

In [16]:
# seis clientes únicos
df['cliente_id'].value_counts()

cliente_id
CLI-A-1    4
CLI-A-4    4
CLI-A-5    4
CLI-A-3    3
CLI-A-2    2
CLI-A-6    2
Name: count, dtype: int64

## data

In [17]:
# as datas estão formatadas todas da mesma forma? Não, há um registro com data = None --> data nao capturada pelo sistema
# o cliente dessa operação é um dos com maior volume de dados na nossa base
df[~df["data"].str.fullmatch(r"\d{4}-\d{2}-\d{2}", na=False)]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
17,OP-0017,CLI-A-5,None,4300.0,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema


In [18]:
# todas são datas válidas? todas as datas, com exceção do caso encontrado, são data válidas!
datas = pd.to_datetime(df["data"], format="%Y-%m-%d", errors="coerce")

df[datas.isna()]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
17,OP-0017,CLI-A-5,None,4300.0,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema


## valor

In [19]:
# não há valores nulos
df[df['valor'].isna()]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao


## moeda

In [20]:
# somente um registro em USD
df['moeda'].value_counts()

moeda
BRL    18
USD     1
Name: count, dtype: int64

## valor_brl

In [21]:
# criamos uma coluna para normalizar valores para BRL
df["valor_brl"] = df["valor"]

df.loc[df["moeda"] == "USD", "valor_brl"] = (
    df.loc[df["moeda"] == "USD", "valor"] * TAXA_CAMBIO
)

C:\Users\yghor\AppData\Local\Temp\ipykernel_6624\184986511.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["valor_brl"] = df["valor"]


## canal

In [22]:
# não há valores anormais
df['canal'].value_counts()

canal
pix        8
ted        5
boleto     3
cartao     2
especie    1
Name: count, dtype: int64

## tipo

In [23]:
# não há valores anormais
df['tipo'].value_counts()

tipo
transferencia_enviada     10
pagamento                  5
transferencia_recebida     3
deposito                   1
Name: count, dtype: int64

## contraparte	

In [24]:
# não há valores anormais
df['contraparte'].value_counts()

contraparte
Alfa Comercio LTDA     4
Delta Transportes      4
Beta Servicos ME       3
Gama Distribuidora     3
Epsilon Consultoria    3
Zeta Importacao        2
Name: count, dtype: int64

# Investigações

## Volume por cliente

In [25]:
volume_cliente = (
    df.groupby("cliente_id")["valor_brl"]
      .sum()
      .reset_index(name="volume_total_brl")
)

volume_cliente

,cliente_id,volume_total_brl
0,CLI-A-1,57500.0
1,CLI-A-2,52900.0
2,CLI-A-3,48500.0
3,CLI-A-4,79500.0
4,CLI-A-5,16900.0
5,CLI-A-6,10200.0


## Operações por Canal

In [26]:
operacoes_canal = (
    df.groupby("canal")
      .size()
      .reset_index(name="quantidade_operacoes")
)

operacoes_canal

,canal,quantidade_operacoes
0,boleto,3
1,cartao,2
2,especie,1
3,pix,8
4,ted,5


## Regra 1 — Fracionamento

In [27]:
# Precisamos saber quantidade de operações, soma dos valores e maior operação para cada par cliente-data
fracionamento = (
    df.groupby(["cliente_id", "data"])
      .agg(
          quantidade_operacoes=("id", "count"),
          soma_valor_brl=("valor_brl", "sum"),
          maior_operacao_brl=("valor_brl", "max")
      )
      .reset_index()
)

# Aplicando a regra
fracionamento["fracionamento"] = (
    (fracionamento["quantidade_operacoes"] >= 3) &
    (fracionamento["soma_valor_brl"] > 50_000) &
    (fracionamento["maior_operacao_brl"] < 20_000)
)

fracionamento

,cliente_id,data,quantidade_operacoes,soma_valor_brl,maior_operacao_brl,fracionamento
0,CLI-A-1,2026-03-09,3,54200.0,18800.0,True
1,CLI-A-1,2026-03-21,1,3300.0,3300.0,False
2,CLI-A-2,2026-03-14,2,52900.0,27000.0,False
3,CLI-A-3,2026-03-05,3,48500.0,17200.0,False
4,CLI-A-4,2026-03-03,1,3800.0,3800.0,False
5,CLI-A-4,2026-03-11,1,5100.0,5100.0,False
6,CLI-A-4,2026-03-18,1,5800.0,5800.0,False
7,CLI-A-4,2026-03-24,1,64800.0,64800.0,False
8,CLI-A-5,2026-03-07,1,2900.0,2900.0,False
9,CLI-A-5,2026-03-16,1,7000.0,7000.0,False


O cliente CLI-A-5 possui uma transação sem data, porém, mesmo incluindo esta na agregação, o mesmo não entraria nos critérios de flag, visto que possui no máximo uma transação por data.

Agora vamos adicionar as flags no df original:

In [28]:
clientes_fracionamento = (
    fracionamento.loc[
        fracionamento["fracionamento"],
        "cliente_id"
    ]
    .unique()
)

df["flag_fracionamento"] = (
    df["cliente_id"].isin(clientes_fracionamento)
)

C:\Users\yghor\AppData\Local\Temp\ipykernel_6624\787473797.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["flag_fracionamento"] = (


#### Tomemos como comparação os casos:
* CLI-A-3 em 2026-03-05
* CLI-A-1 em 2026-03-09

Ambos possuem três operações, mas somente o segundo foi marcado. Apesar do primeiro também ter a maior operação até 20000, a soma dos valores das transações não ultrapassa 50000.

In [29]:
casos = fracionamento[
    (
        (fracionamento["cliente_id"] == "CLI-A-3") &
        (fracionamento["data"] == "2026-03-05")
    )
    |
    (
        (fracionamento["cliente_id"] == "CLI-A-1") &
        (fracionamento["data"] == "2026-03-09")
    )
]

casos

,cliente_id,data,quantidade_operacoes,soma_valor_brl,maior_operacao_brl,fracionamento
0,CLI-A-1,2026-03-09,3,54200.0,18800.0,True
3,CLI-A-3,2026-03-05,3,48500.0,17200.0,False


## Regra 2 - Valor atípico

In [30]:
estatisticas_cliente = (
    df.groupby("cliente_id")["valor_brl"]
      .agg(
          quantidade_operacoes="count",
          mediana_brl="median"
      )
      .reset_index()
)

In [31]:
estatisticas_cliente

,cliente_id,quantidade_operacoes,mediana_brl
0,CLI-A-1,4,17700.0
1,CLI-A-2,2,26450.0
2,CLI-A-3,3,16100.0
3,CLI-A-4,4,5450.0
4,CLI-A-5,4,3600.0
5,CLI-A-6,2,5100.0


In [32]:
df = df.merge(
    estatisticas_cliente,
    on="cliente_id",
    how="left"
)

In [33]:
df[['cliente_id', 'quantidade_operacoes', 'valor_brl', 'mediana_brl']]

,cliente_id,quantidade_operacoes,valor_brl,mediana_brl
0,CLI-A-1,4,18100.0,17700.0
1,CLI-A-1,4,17300.0,17700.0
2,CLI-A-1,4,18800.0,17700.0
3,CLI-A-1,4,3300.0,17700.0
4,CLI-A-2,2,25900.0,26450.0
5,CLI-A-2,2,27000.0,26450.0
6,CLI-A-3,3,17200.0,16100.0
7,CLI-A-3,3,15200.0,16100.0
8,CLI-A-3,3,16100.0,16100.0
9,CLI-A-4,4,3800.0,5450.0


In [34]:
df["5x_mediana_brl"] = 5 * df["mediana_brl"]

df["flag_valor_atipico"] = (
    (df["quantidade_operacoes"] >= 4) &
    (df["valor_brl"] > df["5x_mediana_brl"])
)

In [ ]:
df.loc[
    df["flag_valor_atipico"],
    [
        "id",
        "cliente_id",
        "quantidade_operacoes",
        "mediana_brl",
        "5x_mediana_brl",
        "valor_brl",
        "flag_valor_atipico"
    ]
]

,id,cliente_id,mediana_brl,5x_mediana_brl,valor_brl,quantidade_operacoes,flag_valor_atipico
12,OP-0013,CLI-A-4,5450.0,27250.0,64800.0,4,True


In [40]:
clientes_validacao = ["CLI-A-4", "CLI-A-5"]

validacao = df[
    df["cliente_id"].isin(clientes_validacao)
][[
    "id",
    "cliente_id",
    "quantidade_operacoes",
    "mediana_brl",
    "5x_mediana_brl",
    "valor_brl",
    "flag_valor_atipico"
]].sort_values(["cliente_id", "valor_brl"])

validacao

,id,cliente_id,quantidade_operacoes,mediana_brl,5x_mediana_brl,valor_brl,flag_valor_atipico
9,OP-0010,CLI-A-4,4,5450.0,27250.0,3800.0,False
10,OP-0011,CLI-A-4,4,5450.0,27250.0,5100.0,False
11,OP-0012,CLI-A-4,4,5450.0,27250.0,5800.0,False
12,OP-0013,CLI-A-4,4,5450.0,27250.0,64800.0,True
15,OP-0016,CLI-A-5,4,3600.0,18000.0,2700.0,False
13,OP-0014,CLI-A-5,4,3600.0,18000.0,2900.0,False
16,OP-0017,CLI-A-5,4,3600.0,18000.0,4300.0,False
14,OP-0015,CLI-A-5,4,3600.0,18000.0,7000.0,False
